## Transcript Cleaning and Merging with Company Information

In [ ]:
import pandas as pd
import wrds

# Load the latest table containing company_name
file_name = 'deepseek_company_info_with_cluster.csv'
df_input = pd.read_csv(file_name)

# Make sure the ticker column is clean and all uppercase
ticker_list = df_input['ticker'].dropna().astype(str).str.upper().unique().tolist()
tickers_sql = tuple(ticker_list)

print("Connecting to WRDS database...")
db = wrds.Connection(wrds_username='hiharryyim123')

# Use Ticker for the first round of high-precision matching
sql_query = f"""
    SELECT relatedcompanyid AS companyid, symbolvalue AS ticker
    FROM ciq.ciqsymbol
    WHERE symbolvalue IN {tickers_sql}
      AND activeflag = 1 
"""

try:
    print("Running the first round of Ticker matching...")
    df_mapping = db.raw_sql(sql_query)
    df_clean = df_mapping.drop_duplicates(subset=['ticker'], keep='first').copy()
    
    # Merge the matched IDs
    df_merged = pd.merge(df_input, df_clean, on='ticker', how='left')
    
    # Find N/A values
    df_missing = df_merged[df_merged['companyid'].isnull()]
    
    # Get the successfully matched ID string
    ciq_ids_success = df_merged['companyid'].dropna().astype(int).astype(str).tolist()
    
    print(f"\nFirst round matching done. Total: {len(ticker_list)} companies. Matched: {len(ciq_ids_success)}.")
    
    if not df_missing.empty:
        print(f"\nFound {len(df_missing)} companies that could not be matched by Ticker:")
        # Print missing companies for manual review
        print(df_missing[['ticker', 'company_name']].to_string(index=False))
        print("\nSuggestion: look up the missing company names or Tickers in WRDS 'Code Lookup', then manually add their IDs to the ID string.")
    else:
        print("\nAll companies matched successfully.")

    # Print the successfully matched ID string for downstream use
    print("\nSuccessfully matched ID string (copy directly to download text):")
    print(",".join(ciq_ids_success))
    
    # Save the full result
    output_filename = 'deepseek_company_info_with_ciq_id.csv'
    df_merged.to_csv(output_filename, index=False)
    print(f"\nFull report with IDs saved to: {output_filename}")

except Exception as e:
    print(f"Error occurred: {e}")
finally:
    db.close()
    print("\nDatabase connection closed.")

Connecting to WRDS database...
Loading library list...
Done
Running the first round of Ticker matching...

First round matching done. Total: 531 companies. Matched: 518.

Found 13 companies that could not be matched by Ticker:
ticker                              company_name
  VMEO                               Vimeo, Inc.
  TIXT            TELUS International (Cda) Inc.
 LYTHF     Lytus Technologies Holdings PTV. Ltd.
  OCFT ONECONNECT FINANCIAL TECHNOLOGY CO., LTD.
  XTKG                     X3 Holdings Co., Ltd.
   DAY                            Dayforce, Inc.
  JAMF                        Jamf Holding Corp.
 QVCGA                           QVC Group, Inc.
 NJDCY                                NIDEC CORP
 ATEYY                            ADVANTEST CORP
  SNCR              SYNCHRONOSS TECHNOLOGIES INC
 IFNNY                  INFINEON TECHNOLOGIES AG
  SPNS            SAPIENS INTERNATIONAL CORP N V

Suggestion: look up the missing company names or Tickers in WRDS 'Code Lookup', then m

In [ ]:
#Check duplicates in the final matched universe
import pandas as pd

df_fin = pd.read_csv('deepseek_company_info_with_ciq_id.csv') # Make sure the filename is correct

# Catch duplicate rows
duplicates = df_fin[df_fin.duplicated(subset=['companyid'], keep=False)]
if not duplicates.empty:
    print("Duplicate rows found:")
    print(duplicates[['ticker', 'companyid']]) # Assuming the table has a ticker column, print it to see which ones

# Catch null/invalid rows
missing = df_fin[df_fin['companyid'].isnull()]
if not missing.empty:
    print("\nNull rows found. The following companies have no ID:")
    print(missing)

Duplicate rows found:
    ticker   companyid
66    VMEO         NaN
90    TIXT         NaN
101  LYTHF         NaN
126   OCFT         NaN
141   XTKG         NaN
157    DAY         NaN
159   JAMF         NaN
180    AIP    873964.0
222   CYBR  25460099.0
237     AI    873964.0
331  QVCGA         NaN
343   PANW  25460099.0
366  NJDCY         NaN
367  ATEYY         NaN
371   SNCR         NaN
387  IFNNY         NaN
457   SPNS         NaN

Null rows found. The following companies have no ID:
    ticker                               company_name   sic  \
66    VMEO                                Vimeo, Inc.  7370   
90    TIXT             TELUS International (Cda) Inc.  7374   
101  LYTHF      Lytus Technologies Holdings PTV. Ltd.  7374   
126   OCFT  ONECONNECT FINANCIAL TECHNOLOGY CO., LTD.  7370   
141   XTKG                      X3 Holdings Co., Ltd.  7371   
157    DAY                             Dayforce, Inc.  7372   
159   JAMF                         Jamf Holding Corp.  7372   
331  Q

## Note: Manual completion of missing `companyid`s

The cell above saves the initial matching result as **`deepseek_company_info_with_ciq_id.csv`**. However, 13 companies (e.g. `VMEO`, `TIXT`, `OCFT`, `DAY`, `JAMF`, `NJDCY`, `ATEYY`, `IFNNY`, ...) could not be resolved by ticker alone and were left with `NaN` in the `companyid` column.

I manually looked up these companies in WRDS *Code Lookup* and filled in their `companyid`s by hand, then saved the corrected file as **`deepseek_company_info_with_ciq_id_edited.csv`**. From this point onward, all downstream cells load `deepseek_company_info_with_ciq_id_edited.csv` instead of the original file, so that the final universe reflects the manual corrections.

In [ ]:
import pandas as pd


file_name = 'deepseek_company_info_with_ciq_id_edited.csv'
df_final = pd.read_csv(file_name)


ciq_ids = df_final['companyid'].dropna().astype(int).astype(str).tolist()

# Join  all into one long comma-separated string
id_string_for_website = ",".join(ciq_ids)


print(f"Successfully extracted and joined {len(ciq_ids)} company IDs.\n")
print("Triple-click to select the text below, then copy-paste it into the web form:\n")
print(id_string_for_website)

Successfully extracted and joined 531 company IDs.

Triple-click to select the text below, then copy-paste it into the web form:

127912549,40814475,608901501,1859254185,1865732034,711461819,1838210230,1680095370,1801104960,20333146,704436982,1816580458,1825699846,289284240,1792633815,1888781773,1802801041,763109,879999,82617603,226872615,647543902,36733219,1764297860,422612625,1765795048,586158112,1686418647,601254195,171658800,7688789,117851917,691274020,247542783,1678014439,1679676647,136714,379289324,1678202500,598954721,431954144,20726037,252789627,6939353,874732,671730212,712701220,285799345,1685761306,225024565,1029971,139938259,208444566,214992529,53837907,62400334,73957391,1829179003,1869534890,1680837259,28748353,8951934,52055919,701091442,560353894,270093415,105810806,26258006,717472454,53838219,381848270,706899929,310103275,687352112,704377650,709612693,699650200,706105256,133833909,608737847,1819414146,633790627,250281193,542359809,115222699,674822063,424226705,53569409,59

In [ ]:
import os
import json
import pandas as pd

#the transcript JSON files 
folder_path = '/Users/harrymac/Desktop/Columbia S2/Machine Learning2/Group/Transcript_json'

all_rows = []

print("Parsing JSON files and extracting each executive statement...")

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        file_path = os.path.join(folder_path, filename)
        
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                data = json.load(file)
                
                # Extract meeting-level metadata
                company_id = data.get('companyid')
                company_name = data.get('companyname')
                headline = data.get('headline')
                transcript_date = data.get('transcriptcreationdate')
                
                # Extract the list of dialogue components
                components = data.get('components', [])
                
                # Iterate through each dialogue segment
                for comp in components:
                    row = {
                        'file_name': filename,
                        'companyid': company_id,
                        'companyname': company_name,
                        'headline': headline,
                        'transcript_date': transcript_date,
                        'component_order': comp.get('componentorder'),       
                        'speaker_type': comp.get('componenttypename'),       
                        'speaker_name': comp.get('personname'),              
                        'speaker_company': comp.get('companyofperson'),      
                        'text': str(comp.get('text', '')).replace('\n', ' ').replace('\r', ' ') 
                    }
                    all_rows.append(row)
                    
        except Exception as e:
            print(f"Error reading file {filename}, skipping: {e}")

# Convert to a single DataFrame
df_cleaned = pd.DataFrame(all_rows)


df_executives_only = df_cleaned[df_cleaned['speaker_type'].isin(['Presenter Speech', 'Answer'])].copy()

output_name = 'Executives_Transcripts_Cleaned.csv'
df_executives_only.to_csv(output_name, index=False)

print(f"\nDone. Flattened the data and removed all analyst questions.")
print(f"Clean executive statements table saved as {output_name}, total rows: {len(df_executives_only)}.")
print("\nPreview of the data:")
print(df_executives_only[['companyname', 'speaker_type', 'speaker_name', 'text']].head(3))

Merging the manually completed `companyid`s back into the main table ensures that we have a complete and accurate dataset for all companies, which is crucial for any subsequent analysis or modeling tasks.

In [ ]:
import pandas as pd

df_info = pd.read_csv('deepseek_company_info_with_ciq_id_edited.csv')

# Load the cleaned executive text table 
df_nlp = pd.read_csv('Executives_Transcripts_Cleaned.csv')

# Make sure ID format is unified as numeric
df_info['companyid'] = pd.to_numeric(df_info['companyid'], errors='coerce')
df_nlp['companyid'] = pd.to_numeric(df_nlp['companyid'], errors='coerce')

# Extract the set of all valid unique IDs from the NLP table
valid_nlp_ids = df_nlp['companyid'].dropna().unique()

df_final_universe = df_info[df_info['companyid'].isin(valid_nlp_ids)].copy()

# Export the final result
output_file = 'Final_Matched_Universe_Full.csv'
df_final_universe.to_csv(output_file, index=False)

print(f"Filtering complete.")
print(f"Final list saved as {output_file}, containing {len(df_final_universe)} fully aligned companies.")
print("\nThe table includes all the information you need:")
print(df_final_universe[['ticker', 'company_name', 'companyid', 'cluster']].head(3))

Filtering complete.
Final list saved as Final_Matched_Universe_Full.csv, containing 404 fully aligned companies.

The table includes all the information you need:
  ticker           company_name   companyid  cluster
2     TE         T1 Energy Inc.   608901501        0
3    WAY  Waystar Holding Corp.  1859254185        0
8   NATL        NCR Atleos Corp  1801104960        0
